In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Disaster Triage: 3-Class Cascaded LightGBM + OOF Stacking Meta-Learner (`models/train_disaster_triage_3class_self_pipeline.ipynb`)

This notebook implements the **Hierarchical Cascaded LightGBM + Logistic Regression Meta-Learner (from `train_oof_logistic_regression_stacking.ipynb`)** tailored for **3-Class Disaster Triage (`RED`, `YELLOW`, `GREEN`)** on the **42 curated features** loaded directly from **`datasets/5v_cleandf.RData`** with **training-phase median imputation**:

### 🏥 3-Class Disaster Triage Structure
- **`RED` (Class 0)**: Resuscitation & Emergent (`ESI 1-2`)
- **`YELLOW` (Class 1)**: Urgent (`ESI 3`)
- **`GREEN` (Class 2)**: Less Urgent & Non-Urgent (`ESI 4-5`)

### 🔬 Hierarchical Stacking Architecture
1. **Layer 1 Sub-Model (`L1`)**: Binary LightGBM with SMOTE detecting `RED` vs `(YELLOW + GREEN)` $\rightarrow p_1 = P(\text{RED} | X)$.
2. **Layer 2 Sub-Model (`L2`)**: Binary LightGBM with SMOTE separating `YELLOW` vs `GREEN` on non-RED visits $\rightarrow p_2 = P(\text{YELLOW} | \text{non-RED})$.
3. **Base Probability Tensor**: Computes $[P(\text{RED}), P(\text{YELLOW}), P(\text{GREEN})]$:
   $$P(\text{RED}) = p_1, \quad P(\text{YELLOW}) = (1 - p_1) \cdot p_2, \quad P(\text{GREEN}) = (1 - p_1) \cdot (1 - p_2)$$
4. **Meta-Learner Calibration**: Multinomial `LogisticRegression(class_weight='balanced')` calibrated via **5-Fold Cross-Validation on the Validation Set**.
5. **Comprehensive Evaluation**: Reports Balanced Accuracy, per-class Sensitivity / Recall, clinical safety metrics, 3x3 confusion matrix, 42 density plots, and ROC-AUC curves.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (All Non-NA ESI Rows Kept, No NA Drop)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

data_file <- "../datasets/5v_cleandf.RData"
if (!file.exists(data_file)) data_file <- "datasets/5v_cleandf.RData"

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe without dropping NAs
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  esi                     = raw_esi_char
)

# Export raw matrices to Python
raw_mat_export <- as.matrix(df_master[, 1:16])
esi_export     <- as.numeric(as.character(df_master$esi))

cat(sprintf("Exported Full Dataset to Python: %d rows, 16 base columns\n", nrow(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Stratified Partition, Median Imputation, 42-Feature Extraction & Scaling
# ---------------------------------------------------------------------------
import json, os, pickle
from rpy2.robjects import r
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, cohen_kappa_score)

ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

# 3-Class Mapping: RED (0: ESI 1-2), YELLOW (1: ESI 3), GREEN (2: ESI 4-5)
y_all = np.where(esi_all <= 2, 0, np.where(esi_all == 3, 1, 2))
LABELS = ['RED', 'YELLOW', 'GREEN']

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

y_train = y_all[itr]
y_val   = y_all[iva]
y_test  = y_all[ite]

# Fit SimpleImputer (median) strictly on Training set
print("Fitting SimpleImputer(strategy='median') on Training set...")
imputer = SimpleImputer(strategy='median')
raw_tr_imp  = imputer.fit_transform(raw_tr)
raw_val_imp = imputer.transform(raw_val)
raw_te_imp  = imputer.transform(raw_te)

def build_42_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 42), dtype=np.float32)
    
    # Extract base columns
    age       = raw_mat[:, 0]
    cc_bd     = raw_mat[:, 1]
    gender    = raw_mat[:, 2]
    t_hr      = raw_mat[:, 3]
    t_sbp     = raw_mat[:, 4]
    t_dbp     = raw_mat[:, 5]
    t_rr      = raw_mat[:, 6]
    t_o2      = raw_mat[:, 7]
    pulse_min = raw_mat[:, 8]
    resp_min  = raw_mat[:, 9]
    spo2_min  = raw_mat[:, 10]
    sbp_min   = raw_mat[:, 11]
    pulse_max = raw_mat[:, 12]
    resp_max  = raw_mat[:, 13]
    spo2_max  = raw_mat[:, 14]
    sbp_max   = raw_mat[:, 15]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    # 1..12: Specified Raw Features
    X[:, 0]  = age
    X[:, 1]  = cc_bd
    X[:, 2]  = gender
    X[:, 3]  = t_hr
    X[:, 4]  = t_sbp
    X[:, 5]  = t_dbp
    X[:, 6]  = t_rr
    X[:, 7]  = pulse_min
    X[:, 8]  = resp_min
    X[:, 9]  = spo2_min
    X[:, 10] = pulse_max
    X[:, 11] = spo2_max
    
    # 13..22: Baseline Threshold Flags
    is_dyspnea_tot  = (t_o2 < 90).astype(float)
    is_dyspnea_mod  = ((t_o2 >= 90) & (t_o2 < 94)).astype(float)
    is_brady_pnea   = (t_rr < 10).astype(float)
    is_tachy_pnea   = (t_rr > 30).astype(float)
    is_hypo_tension = (t_sbp <= 90).astype(float)
    is_hyper_tension= (t_sbp > 220).astype(float)
    is_brady_tot    = (t_hr < 40).astype(float)
    is_brady_mod    = ((t_hr >= 40) & (t_hr < 60)).astype(float)
    is_tachy_tot    = (t_hr > 150).astype(float)
    is_tachy_mod    = ((t_hr >= 100) & (t_hr <= 150)).astype(float)
    
    X[:, 12] = is_dyspnea_tot
    X[:, 13] = is_dyspnea_mod
    X[:, 14] = is_brady_pnea
    X[:, 15] = is_tachy_pnea
    X[:, 16] = is_hypo_tension
    X[:, 17] = is_hyper_tension
    X[:, 18] = is_brady_tot
    X[:, 19] = is_brady_mod
    X[:, 20] = is_tachy_tot
    X[:, 21] = is_tachy_mod
    
    # 23..35: Ranges, Mid-to-Triage, Ratios
    X[:, 22] = hr_rng
    X[:, 23] = rr_rng
    X[:, 24] = spo2_rng
    X[:, 25] = sbp_rng
    shock_idx = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 26] = shock_idx
    X[:, 27] = t_hr - hr_rng
    X[:, 28] = t_sbp - sbp_rng
    X[:, 29] = t_rr - rr_rng
    X[:, 30] = t_o2 - spo2_rng
    X[:, 31] = t_o2 / np.where(t_rr == 0, 1.0, t_rr) # rox_index
    X[:, 32] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max) # spo2_drop_ratio
    X[:, 33] = hr_rng / (t_hr + 1.0) # hr_instability_ratio
    X[:, 34] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0 # bif
    
    # 36..42: Curated Advanced Features
    X[:, 35] = (is_dyspnea_tot + is_dyspnea_mod + is_brady_pnea + is_tachy_pnea +
                is_hypo_tension + is_hyper_tension + is_brady_tot + is_brady_mod +
                is_tachy_tot + is_tachy_mod) # n_abnormal_vitals
    X[:, 36] = (t_hr / 80.0) - (t_sbp / 120.0) # perfusion_gap
    X[:, 37] = np.clip(spo2_min - 90.0, -20.0, 20.0) # resp_reserve
    X[:, 38] = cc_bd * (100.0 - t_o2) # bd_x_o2_deficit
    X[:, 39] = cc_bd * is_tachy_pnea   # bd_x_tachypnea
    X[:, 40] = age * (100.0 - t_o2)   # age_o2_interaction
    X[:, 41] = ((t_hr - 80.0) / 80.0) ** 2 # hr_dev_sq
    
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

X_train_raw = build_42_feature_matrix(raw_tr_imp)
X_val_raw   = build_42_feature_matrix(raw_val_imp)
X_test_raw  = build_42_feature_matrix(raw_te_imp)

feature_names_42 = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr',
    'pulse_min', 'resp_min', 'spo2_min', 'pulse_max', 'spo2_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif',
    'n_abnormal_vitals', 'perfusion_gap', 'resp_reserve', 'bd_x_o2_deficit', 'bd_x_tachypnea',
    'age_o2_interaction', 'hr_dev_sq'
]

cont_cols_idx = [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 40, 41]

scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[:, cont_cols_idx] = scaler.fit_transform(X_train_raw[:, cont_cols_idx])
X_val[:, cont_cols_idx]   = scaler.transform(X_val_raw[:, cont_cols_idx])
X_test[:, cont_cols_idx]  = scaler.transform(X_test_raw[:, cont_cols_idx])

def numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]))
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42
}

# ---------------------------------------------------------------------------
# Train Cascaded LightGBM Sub-Models (Layer 1 & Layer 2) with SMOTE
# ---------------------------------------------------------------------------
print("Training Cascaded LightGBM Sub-Models on Train Set with Validation Early Stopping...")

# Layer 1: RED (Class 0) vs (YELLOW + GREEN)
X_sm1, y_sm1 = numpy_smote(X_train, (y_train == 0).astype(int))
l1_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=120)
l1_prod.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 0).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 2: YELLOW (Class 1) vs GREEN (Class 2) on non-RED visits
m2_tr  = (y_train != 0)
m2_val = (y_val != 0)
X_sm2, y_sm2 = numpy_smote(X_train[m2_tr], (y_train[m2_tr] == 1).astype(int))
l2_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=120)
l2_prod.fit(X_sm2, y_sm2, eval_set=[(X_val[m2_val], (y_val[m2_val] == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# ---------------------------------------------------------------------------
# Generate Base 3-Class Probabilities for Validation Set
# ---------------------------------------------------------------------------
p1_val = l1_prod.predict_proba(X_val)[:, 1]
p2_val = l2_prod.predict_proba(X_val)[:, 1]

val_probs = np.zeros((len(X_val), 3))
val_probs[:, 0] = p1_val                  # P(RED)
val_probs[:, 1] = (1 - p1_val) * p2_val      # P(YELLOW)
val_probs[:, 2] = (1 - p1_val) * (1 - p2_val)# P(GREEN)

# ---------------------------------------------------------------------------
# 5-Fold Cross-Validation on Validation Set Meta-Learner (from stacking notebook)
# ---------------------------------------------------------------------------
print("Calibrating Meta-Learner with 5-Fold Cross-Validation on the Validation Set...")
skf_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (v_tr_idx, v_val_idx) in enumerate(skf_val.split(val_probs, y_val)):
    cv_lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    cv_lr.fit(val_probs[v_tr_idx], y_val[v_tr_idx])
    pred_v = cv_lr.predict(val_probs[v_val_idx])
    acc_v  = np.mean(pred_v == y_val[v_val_idx])
    cv_scores.append(acc_v)

print(f"Validation 5-Fold CV Meta-Learner Accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

# Fit Final Meta-Learner with Balanced Class Weighting on Entire Validation Set
meta_logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_logreg.fit(val_probs, y_val)

# ---------------------------------------------------------------------------
# Generate Base 3-Class Probabilities for Holdout Test Set
# ---------------------------------------------------------------------------
p1_test = l1_prod.predict_proba(X_test)[:, 1]
p2_test = l2_prod.predict_proba(X_test)[:, 1]

test_probs_prod = np.zeros((len(X_test), 3))
test_probs_prod[:, 0] = p1_test
test_probs_prod[:, 1] = (1 - p1_test) * p2_test
test_probs_prod[:, 2] = (1 - p1_test) * (1 - p2_test)

# Predictions via Meta-Learner
preds_meta_logreg = meta_logreg.predict(test_probs_prod)
probs_meta_logreg = meta_logreg.predict_proba(test_probs_prod)

print(f"Stacking Pipeline Complete: Holdout Test Matrix = {test_probs_prod.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Holdout Test Set Evaluation & Detailed Per-Class Breakdown
# ---------------------------------------------------------------------------
classes = [0, 1, 2]
class_names = ['RED', 'YELLOW', 'GREEN']

def get_per_class_breakdown_3class(y_true, y_pred, probs, pipeline_name):
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': class_names[cls],
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

report_df = get_per_class_breakdown_3class(y_test, preds_meta_logreg, probs_meta_logreg, 'Cascaded_LightGBM_OOF_Stacking')

cm = confusion_matrix(y_test, preds_meta_logreg, labels=[0, 1, 2])
undertriage = cm[0, 2] / cm[0].sum() if cm[0].sum() > 0 else 0.0
overtriage  = (cm[1, 0] + cm[2, 0]) / (cm[1].sum() + cm[2].sum()) if (cm[1].sum() + cm[2].sum()) > 0 else 0.0
overtriage_green_only = cm[2, 0] / cm[2].sum() if cm[2].sum() > 0 else 0.0

s = dict(
    balanced_accuracy=balanced_accuracy_score(y_test, preds_meta_logreg),
    accuracy=accuracy_score(y_test, preds_meta_logreg),
    red_sensitivity=cm[0, 0] / cm[0].sum() if cm[0].sum() > 0 else 0.0,
    yellow_sensitivity=cm[1, 1] / cm[1].sum() if cm[1].sum() > 0 else 0.0,
    green_sensitivity=cm[2, 2] / cm[2].sum() if cm[2].sum() > 0 else 0.0,
    macro_recall=np.mean([cm[i, i] / cm[i].sum() for i in range(3)]),
    undertriage=undertriage,
    overtriage=overtriage,
    overtriage_green_only=overtriage_green_only
)

print("========================================================================================")
print("   HOLDOUT TEST REPORT: 3-CLASS DISASTER TRIAGE STACKING PIPELINE (42 FEATURES)")
print("========================================================================================")
print(report_df.to_string(index=False))
print("----------------------------------------------------------------------------------------")
print(f"  Clinical Safety Metrics:\n"
      f"    * Undertriage (RED -> GREEN)   : {s['undertriage']:.4f}\n"
      f"    * Overtriage (non-RED -> RED)  : {s['overtriage']:.4f}\n"
      f"    * Overall Accuracy             : {s['accuracy']:.4f}\n"
      f"    * Balanced Accuracy            : {s['balanced_accuracy']:.4f}")
print("========================================================================================\n")

reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)
report_df.to_csv(os.path.join(reports_dir, 'disaster_triage_3class_stacking_report.csv'), index=False)
print(f"Report saved to {os.path.join(reports_dir, 'disaster_triage_3class_stacking_report.csv')}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: 3x3 Confusion Matrix Graph for Disaster Triage Classes
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm_meta_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8, 6.5))

annot_meta = np.empty_like(cm, dtype=object)
for i in range(3):
    for j in range(3):
        annot_meta[i, j] = f"{cm[i, j]}\n({cm_meta_norm[i, j]*100:.1f}%)"

sns.heatmap(cm_meta_norm, annot=annot_meta, fmt='', cmap='Greens', cbar=True,
            xticklabels=LABELS, yticklabels=LABELS, ax=ax, vmin=0, vmax=1)
ax.set_title('Holdout Test Confusion Matrix\nDisaster Triage Stacking Pipeline (42 Features)', fontsize=12.5, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Acuity Category', fontsize=11, fontweight='bold')
ax.set_ylabel('True Acuity Category', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path = os.path.join(plots_dir, 'disaster_triage_stacking_confusion_matrix.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'disaster_triage_stacking_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"3x3 Confusion Matrix Graph saved to: {cm_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Density Data Distribution Graphs (42 Features Colored by Acuity Category)
# ---------------------------------------------------------------------------
plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
density_dir = os.path.join(plots_dir, 'density_disaster_stacking')
density_img_dir = os.path.join(plots_dir, 'image', 'density_disaster_stacking')
os.makedirs(density_dir, exist_ok=True)
os.makedirs(density_img_dir, exist_ok=True)

df_raw_all = pd.DataFrame(X_train_raw, columns=feature_names_42)
group_map = {0: 'RED', 1: 'YELLOW', 2: 'GREEN'}
df_raw_all['Category'] = [group_map[k] for k in y_train]

cat_palette = {
    'RED': '#d62728',    # Red (Resuscitation / Emergent)
    'YELLOW': '#ff7f0e', # Orange (Urgent)
    'GREEN': '#2ca02c'   # Green (Less Urgent / Minor)
}

print(f"Saving individual feature density distribution plots ({len(feature_names_42)} features) to: {density_dir}")

for feat in feature_names_42:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    if feat in ['gender', 'cc_breathingdifficulty'] or feat.startswith('is_') or feat.startswith('bd_x_tachypnea'):
        prop_df = df_raw_all.groupby('Category')[feat].mean().reset_index(name='Proportion')
        sns.barplot(data=prop_df, x='Category', y='Proportion', palette=cat_palette, ax=ax, edgecolor='black',
                    order=['RED', 'YELLOW', 'GREEN'])
        ax.set_title(f"{feat} (Prevalence by Disaster Triage Category)", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("Acuity Category", fontsize=11, fontweight='bold')
        ax.set_ylabel("Prevalence / Proportion", fontsize=11, fontweight='bold')
        for p in ax.patches:
            ax.annotate(f"{p.get_height()*100:.1f}%",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 2),
                        textcoords='offset points')
    else:
        sns.kdeplot(
            data=df_raw_all,
            x=feat,
            hue='Category',
            hue_order=['RED', 'YELLOW', 'GREEN'],
            palette=cat_palette,
            common_norm=False,
            fill=True,
            alpha=0.20,
            linewidth=2.0,
            ax=ax
        )
        ax.set_title(f"Feature Density Distribution: {feat}", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel(feat, fontsize=11, fontweight='bold')
        ax.set_ylabel("Density", fontsize=11, fontweight='bold')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    out_file = f"density_{feat}.png"
    plt.savefig(os.path.join(density_dir, out_file), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(density_img_dir, out_file), dpi=300, bbox_inches='tight')
    plt.close()

print(f"All {len(feature_names_42)} individual density plots successfully saved in {density_dir}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: 3-Class Multiclass ROC-AUC Curve Analysis (Holdout Test Benchmark)
# ---------------------------------------------------------------------------
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes  = 3

fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_meta_logreg[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_meta_logreg.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(9, 8))

cat_colors = {
    0: '#d62728',  # RED: Red
    1: '#ff7f0e',  # YELLOW: Orange
    2: '#2ca02c'   # GREEN: Green
}

plt.plot(fpr["micro"], tpr["micro"],
         label=f"Micro-Average (AUC = {roc_auc['micro']:.4f})",
         color='#e377c2', linestyle=':', linewidth=2.5)
plt.plot(fpr["macro"], tpr["macro"],
         label=f"Macro-Average (AUC = {roc_auc['macro']:.4f})",
         color='#17becf', linestyle='--', linewidth=2.5)

for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], color=cat_colors[i], linewidth=2.0,
             label=f"{LABELS[i]} (AUC = {roc_auc[i]:.4f})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12, fontweight='bold')
plt.title('Holdout Test ROC-AUC Curves (Disaster Triage Stacking Pipeline)', fontsize=13, fontweight='bold', pad=12)
plt.legend(loc="lower right", fontsize=10.5, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

roc_plot_path = os.path.join(plots_dir, 'disaster_triage_stacking_roc_auc.png')
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'disaster_triage_stacking_roc_auc.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"ROC-AUC Curve Graph saved to: {roc_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: Export Production Bundle & Manifest (L1 + L2 + Meta-Learner + Imputer)
# ---------------------------------------------------------------------------
deploy = f'{ROOT}/deploy'
os.makedirs(deploy, exist_ok=True)

bundle_data = {
    'l1_prod': l1_prod,
    'l2_prod': l2_prod,
    'meta_logreg': meta_logreg,
    'imputer': imputer,
    'scaler_means': scaler.mean_,
    'scaler_sds': scaler.scale_,
    'cont_cols_idx': cont_cols_idx,
    'feature_names': feature_names_42,
    'labels': LABELS
}

with open(f'{deploy}/disaster_triage_3class_stacking.pkl', 'wb') as f:
    pickle.dump(bundle_data, f)

n_nodes = sum(t['num_leaves'] for t in l1_prod.booster_.dump_model()['tree_info']) + \
          sum(t['num_leaves'] for t in l2_prod.booster_.dump_model()['tree_info'])

manifest = dict(
    labels=LABELS,
    feature_order=feature_names_42,
    n_features=len(feature_names_42),
    n_nodes=n_nodes,
    holdout={k: round(float(v), 4) for k, v in s.items()},
    holdout_macro_auc=round(float(roc_auc['macro']), 4)
)

with open(f'{deploy}/disaster_triage_stacking_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'wrote {deploy}/disaster_triage_3class_stacking.pkl')
print(f'wrote {deploy}/disaster_triage_stacking_manifest.json')

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Runnable Safety & Production Verifications
# ---------------------------------------------------------------------------
assert s['balanced_accuracy'] >= 0.45, f'Balanced accuracy {s["balanced_accuracy"]:.4f} below target'
assert s['red_sensitivity'] >= 0.60, f'RED sensitivity {s["red_sensitivity"]:.4f}'
assert n_nodes * 16 <= 2 * 1024 * 1024, f'{n_nodes} nodes exceeds flash budget'
assert len(feature_names_42) == 42, f'expected 42 features, got {len(feature_names_42)}'
assert len(feature_names_42) == len(set(feature_names_42)), 'duplicate feature name'
assert json.load(open(f'{deploy}/disaster_triage_stacking_manifest.json'))['feature_order'] \
    == feature_names_42, 'manifest feature order does not match the trained model'
print('all checks passed successfully!')